# PAI最終課題：小型LLMによる長期タスク分解はVLAの実行を助けるか？
## Notebook 3：正しい言い回しでの追加評価（`oracle_grounded`）

**目的**：Notebook 2の `skill_grounded` では、小型LLMがスキルの選択を6回中3回誤ったため、
「学習時の言い回しに合わせたら実行できるか」を正しく測れませんでした。
そこで、**人間が正しく選んだ学習済みの指示文**をサブタスクとして渡し、
「言い回しの問題か、シーン（言い回し×シーンの組み合わせ）の問題か」を切り分けます。

| 条件 | SmolVLAに渡す指示 |
|---|---|
| `oracle_grounded`（本ノートブック） | 各サブタスクを、人間が選んだ正しい学習済み指示文（LIBERO-Objectの言い回し）にして順に |

- 対象：LIBERO-10のタスク0・1・7（タスク4のマグカップには対応する学習済み指示がないため除外）
- 各5エピソード（Notebook 1・2と同じ初期状態0〜4）＝15エピソード、約30分
- Notebook 1・2と**同じDriveフォルダ（`pai_final/`）**の `results.jsonl` に追記し、最後に全条件を比較します
- 新しいセッションから上から順に実行すれば動きます（LLMは使いません）

## 1. GPU確認

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## 2. インストール
再現性のため、LeRobotは動作確認したコミットに固定しています。

`lerobot[libero]` をそのまま入れると、依存パッケージ `egl_probe` / `hf-egl-probe` のビルドがColab上で失敗します。
この2つ（と、それを要求する `robomimic`）はLIBEROの学習用コード（`libero.lifelong`）でしか使われず、
シミュレーション環境の実行には不要なので、**LIBERO本体を依存なしで入れ、環境の実行に必要な依存だけを追加**します。

In [ ]:
LEROBOT_COMMIT = "3f797de5952432284a7c77ad2760218cb74e2f26"  # 2026-10-30時点のmain

# (1) LeRobot本体＋SmolVLA（libero extraは使わない）
!pip -q install "lerobot[smolvla,dataset,scipy-dep] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_COMMIT}" imageio imageio-ffmpeg
# (2) LIBERO本体は依存なしでインストール（egl_probe / robomimic を回避）
!pip -q install --no-deps "hf-libero>=0.1.4,<0.2.0"
# (3) LIBEROの環境実行に必要な依存だけを追加
!pip -q install "robosuite==1.4.0" "bddl==1.0.1" "mujoco>=3.0.0,<3.9.0" easydict h5py termcolor cloudpickle future matplotlib "gymnasium>=0.29.0"

!pip list 2>/dev/null | grep -i -E "^(lerobot|hf-libero|robosuite|bddl|mujoco|torch|transformers|numpy) "

## 3. LIBEROの設定ファイル作成・描画設定
LIBEROは初回import時に対話入力（Y/N）を求めるため、設定ファイルを先に作成しておきます。
描画はヘッドレス環境向けにEGLを使います。

In [ ]:
import os, importlib.util, pathlib, yaml

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

spec = importlib.util.find_spec("libero")
libero_root = pathlib.Path(list(spec.submodule_search_locations)[0]) / "libero"
cfg_dir = pathlib.Path(os.environ.get("LIBERO_CONFIG_PATH", pathlib.Path.home() / ".libero"))
cfg_dir.mkdir(parents=True, exist_ok=True)
paths = {
    "benchmark_root": str(libero_root),
    "bddl_files": str(libero_root / "bddl_files"),
    "init_states": str(libero_root / "init_files"),
    "datasets": str(libero_root.parent / "datasets"),
    "assets": str(libero_root / "assets"),
}
with open(cfg_dir / "config.yaml", "w") as f:
    yaml.dump(paths, f)
print(open(cfg_dir / "config.yaml").read())

from libero.libero import benchmark  # 対話入力なしでimportできればOK
print("LIBERO suites:", sorted(benchmark.get_benchmark_dict().keys()))

## 4. Google Driveのマウントと保存先
無料枠ではGPUが途中で切断されることがあるため、結果は1エピソードごとにDriveへ保存します。
Driveを使わない場合は `USE_DRIVE = False` にしてください（その場合、切断時に結果が消えます）。

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = pathlib.Path("/content/drive/MyDrive/pai_final")
else:
    OUT_DIR = pathlib.Path("/content/pai_final")

(OUT_DIR / "videos").mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT_DIR / "results.jsonl"
print("保存先:", OUT_DIR)

## 5. 実験設定
- **ポリシー**：LeRobot公式CIでLIBERO評価に使われている設定（カメラ名 `camera1`/`camera2`、`empty_cameras=1`）をそのまま使います。
  スモークテストで成功しない場合は、`POLICY_PRESET = "hfvla"` に切り替えて再度セル5〜6を実行してください。
- **タスク**：LIBERO-10のうち「2つの物体を順に操作する」タスク4つ

In [ ]:
POLICY_PRESET = "lerobot_ci"   # "lerobot_ci" または "hfvla"

PRESETS = {
    "lerobot_ci": dict(
        path="lerobot/smolvla_libero",
        camera_name_mapping={"agentview_image": "camera1", "robot0_eye_in_hand_image": "camera2"},
        policy_overrides=["--empty_cameras=1"],
    ),
    "hfvla": dict(
        path="HuggingFaceVLA/smolvla_libero",
        camera_name_mapping=None,  # 既定（image / image2）
        policy_overrides=[],
    ),
}
P = PRESETS[POLICY_PRESET]

SUITE = "libero_10"
TASK_IDS = [0, 1, 4, 7]   # すべて「2つの物体」を扱うタスク
N_EPISODES = 5            # 1タスクあたりのエピソード数（初期状態0〜4を固定で使用）
SEED = 1000
DEVICE = "cuda"
print(P)

## 6. oracle_grounded の指示（人間が選んだ学習済み指示文）
smolvla_liberoがLIBERO-Objectで学習した言い回し「pick up the X and place it in the basket」をそのまま使います。
LIBERO-Objectでの物体名は「cream cheese」（LIBERO-10の「cream cheese box」とは表記が違う）なので、学習時の表記に合わせています。

In [ ]:
ORACLE = {
    0: ["pick up the alphabet soup and place it in the basket", "pick up the tomato sauce and place it in the basket"],
    1: ["pick up the cream cheese and place it in the basket", "pick up the butter and place it in the basket"],
    7: ["pick up the alphabet soup and place it in the basket", "pick up the cream cheese and place it in the basket"],
}
TASK_IDS = list(ORACLE)   # タスク4は対応する学習済み指示がないため除外

# 学習済み指示文に本当に含まれているかを確認
from libero.libero import benchmark as _bm
_obj = _bm.get_benchmark_dict()["libero_object"]()
_learned = {_obj.get_task(i).language for i in range(len(_obj.tasks))}
for tid, steps in ORACLE.items():
    for s in steps:
        print(f"[{tid}] {'OK ' if s in _learned else 'NG '} {s}")

## 7. ポリシーと前処理パイプラインの読み込み
公式 `lerobot-eval` と同じ手順（`make_policy` → `make_pre_post_processors` → `make_env_pre_post_processors`）で組み立てます。

In [ ]:
from pathlib import Path
import numpy as np, torch, gymnasium as gym

from lerobot.configs import PreTrainedConfig
from lerobot.envs import make_env_pre_post_processors, preprocess_observation
from lerobot.envs.configs import LiberoEnv as LiberoEnvConfig
from lerobot.envs import libero as libero_mod
from lerobot.policies import make_policy, make_pre_post_processors
from lerobot.processor import policy_output_to_transition, transition_to_policy_action
from lerobot.utils.constants import ACTION

torch.manual_seed(SEED); np.random.seed(SEED)

env_cfg = LiberoEnvConfig(task=SUITE, task_ids=TASK_IDS, camera_name_mapping=P["camera_name_mapping"])

policy_cfg = PreTrainedConfig.from_pretrained(
    P["path"], cli_overrides=P["policy_overrides"] + [f"--device={DEVICE}"]
)
policy_cfg.pretrained_path = Path(P["path"])

policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg)
policy.eval()

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy_cfg,
    pretrained_path=policy_cfg.pretrained_path,
    preprocessor_overrides={"device_processor": {"device": str(policy.config.device)}},
)
env_preprocessor, env_postprocessor = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)

print("input features :", list(policy.config.input_features.keys()))
print("n_action_steps :", policy.config.n_action_steps, "/ chunk_size:", policy.config.chunk_size)
print(f"GPU memory     : {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 8. 評価ループの定義
`run_episode(task_id, instructions, episode_index)` は、**ベースライン（指示1つ）とLLM分解（指示が複数）の両方で共通**に使います。

- **サブタスクの切り替え規則**：各ステップでLIBEROのゴール述語（BDDLの `:goal` に並ぶ条件）を個別に評価し、満たされた述語の数 *k* に応じて *k* 番目の指示を渡します。
  これはシミュレータの内部状態を使う「オラクル判定」なので、レポートでは限界として明記します。
- 指示が切り替わったら `policy.reset()` で行動チャンクのキューを破棄し、新しい指示で推論し直します。
- **初期状態の固定**：`episode_index` ごとにLIBEROの初期状態を固定するため、条件間で同じ初期配置を比較できます。
- 部分達成（満たした述語の数）も記録し、「どこまで進んだか」も分析できるようにします。

In [ ]:
import time, json, imageio
from PIL import Image, ImageDraw, ImageFont

_SUITE_OBJ = libero_mod._get_suite(SUITE)
_GYM_KW = {k: v for k, v in env_cfg.gym_kwargs.items() if k != "task_ids"}

def make_vec_env(task_id: int, episode_index: int):
    def _fn():
        return libero_mod.LiberoEnv(
            task_suite=_SUITE_OBJ, task_id=task_id, task_suite_name=SUITE,
            camera_name=env_cfg.camera_name, init_states=env_cfg.init_states,
            episode_length=env_cfg.episode_length, episode_index=episode_index, n_envs=1,
            control_mode=env_cfg.control_mode, camera_name_mapping=env_cfg.camera_name_mapping,
            **_GYM_KW,
        )
    return gym.vector.SyncVectorEnv([_fn], autoreset_mode=gym.vector.AutoresetMode.NEXT_STEP)

def task_language(task_id: int) -> str:
    return _SUITE_OBJ.get_task(task_id).language

try:
    _FONT = ImageFont.load_default(size=15)
except TypeError:
    _FONT = ImageFont.load_default()

def annotate(frame: np.ndarray, lines: list[str], scale: int = 2) -> np.ndarray:
    img = Image.fromarray(frame).resize((frame.shape[1] * scale, frame.shape[0] * scale), Image.BICUBIC)
    draw = ImageDraw.Draw(img)
    pad, lh = 6, 19
    draw.rectangle([0, 0, img.width, pad * 2 + lh * len(lines)], fill=(0, 0, 0))
    for i, t in enumerate(lines):
        draw.text((pad, pad + i * lh), t, fill=(255, 255, 255), font=_FONT)
    return np.asarray(img)

@torch.no_grad()
def run_episode(task_id: int, instructions: list[str], episode_index: int,
                condition: str = "baseline", video_path: str | None = None) -> dict:
    t0 = time.time()
    venv = make_vec_env(task_id, episode_index)
    obs, _ = venv.reset(seed=SEED + episode_index)
    lenv = venv.envs[0]
    inner = lenv._env.env                          # LIBEROの問題ドメイン（BDDL）
    goals = inner.parsed_problem["goal_state"]     # 例: [['in','alphabet_soup_1','basket_1_contain_region'], ...]
    max_steps = lenv._max_episode_steps

    policy.reset()
    cur, switches, frames = 0, [], []
    best_sat, success, steps = 0, False, 0

    def eval_goals():
        return [bool(inner._eval_predicate(g)) for g in goals]

    for step in range(max_steps):
        sat = eval_goals()
        n_sat = sum(sat); best_sat = max(best_sat, n_sat)
        target = min(n_sat, len(instructions) - 1)
        if target != cur:
            cur = target
            policy.reset()
            switches.append({"step": step, "to": cur})

        o = preprocess_observation(obs)
        o["task"] = [instructions[cur]]
        o = env_preprocessor(o)
        o = preprocessor(o)
        with torch.inference_mode():
            act = transition_to_policy_action(policy_output_to_transition(policy.select_action(o)))
        act = postprocessor(act)
        act = env_postprocessor({ACTION: act})[ACTION]

        obs, _, terminated, truncated, info = venv.step(act.to("cpu").numpy())
        steps = step + 1

        if video_path is not None:
            frames.append(annotate(lenv.render(), [
                f"[{condition}] task {task_id} / ep {episode_index} / step {steps}",
                f"instruction {cur + 1}/{len(instructions)}: {instructions[cur]}",
                f"goals satisfied: {n_sat}/{len(goals)}",
            ]))

        if bool(np.asarray(terminated)[0]) or bool(np.asarray(truncated)[0]):
            success = bool(np.asarray(info.get("is_success", [False]))[0])
            break

    final_sat = eval_goals()
    success = success or all(final_sat)
    venv.close()

    if video_path is not None and frames:
        imageio.mimsave(video_path, frames, fps=env_cfg.fps)

    return {
        "task_id": task_id, "task": task_language(task_id), "condition": condition,
        "episode": episode_index, "instructions": instructions,
        "success": success, "steps": steps, "max_steps": max_steps,
        "n_goals": len(goals), "best_goals_satisfied": max(best_sat, sum(final_sat)),
        "final_goals_satisfied": sum(final_sat), "switches": switches,
        "wall_sec": round(time.time() - t0, 1), "policy": P["path"],
    }

## 9. 対象タスクとゴール述語の確認
LLM分解のプロンプト設計（Notebook 2）でも使うので、各タスクの指示文と述語を確認しておきます。

In [ ]:
for tid in TASK_IDS:
    venv = make_vec_env(tid, 0); venv.reset(seed=SEED)
    goals = venv.envs[0]._env.env.parsed_problem["goal_state"]
    print(f"[{tid}] {task_language(tid)}")
    for g in goals:
        print("      -", g)
    venv.close()

## 10. oracle_grounded の本番評価
途中で切断されても、このセルを再実行すれば続きから再開します。
動画用に、全エピソードの動画を保存します（`videos/oracle_grounded_t*_e*.mp4`）。

In [ ]:
def load_done(path) -> set:
    done = set()
    if Path(path).exists():
        for line in open(path):
            r = json.loads(line)
            done.add((r["condition"], r["task_id"], r["episode"]))
    return done

def run_condition(condition: str, instr_fn, save_video_eps=(0,)):
    done = load_done(RESULTS_PATH)
    for tid in TASK_IDS:
        for ep in range(N_EPISODES):
            if (condition, tid, ep) in done:
                continue
            vp = str(OUT_DIR / "videos" / f"{condition}_t{tid}_e{ep}.mp4") if ep in save_video_eps else None
            r = run_episode(tid, instr_fn(tid), ep, condition=condition, video_path=vp)
            with open(RESULTS_PATH, "a") as f:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
            print(f"[{condition}] task {tid} ep {ep}: success={r['success']} "
                  f"goals={r['best_goals_satisfied']}/{r['n_goals']} steps={r['steps']} ({r['wall_sec']}s)")

run_condition("oracle_grounded", lambda tid: ORACLE[tid], save_video_eps=tuple(range(N_EPISODES)))

## 11. 全条件の比較・集計
`oracle_grounded` はタスク0・1・7のみなので、**同じ3タスクに絞った比較表**も出します。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_json(RESULTS_PATH, lines=True)
df = df[df["condition"] != "manual_test"].copy()
df["progress"] = df["best_goals_satisfied"] / df["n_goals"]
order = [c for c in ["baseline", "llm_decomp", "skill_grounded", "oracle_grounded"] if c in set(df["condition"])]

overall = (df.groupby("condition").agg(n=("success", "size"), success_rate=("success", "mean"),
                                        mean_progress=("progress", "mean"), mean_steps=("steps", "mean"))
             .reindex(order))
print("=== 全体 ===")
display(overall.style.format({"success_rate": "{:.0%}", "mean_progress": "{:.0%}", "mean_steps": "{:.0f}"}))

by_task = (df.groupby(["task_id", "condition"]).agg(n=("success", "size"), success_rate=("success", "mean"),
                                                    mean_progress=("progress", "mean"), mean_steps=("steps", "mean"))
             .reset_index())
print("=== タスク別 ===")
display(by_task.style.format({"success_rate": "{:.0%}", "mean_progress": "{:.0%}", "mean_steps": "{:.0f}"}))
by_task.to_csv(OUT_DIR / "summary_by_task.csv", index=False)
overall.to_csv(OUT_DIR / "summary_overall.csv")

# 同じ初期状態でのペア比較（baseline vs 各分解条件）
base = df[df["condition"] == "baseline"].set_index(["task_id", "episode"])
rows = []
for cond in order[1:]:
    other = df[df["condition"] == cond].set_index(["task_id", "episode"])
    j = base[["success", "best_goals_satisfied"]].join(other[["success", "best_goals_satisfied"]],
                                                       lsuffix="_base", rsuffix="_dec", how="inner")
    rows.append({"condition": cond, "paired_episodes": len(j),
                 "改善 (失敗→成功)": int((~j.success_base & j.success_dec).sum()),
                 "悪化 (成功→失敗)": int((j.success_base & ~j.success_dec).sum()),
                 "両方成功": int((j.success_base & j.success_dec).sum()),
                 "両方失敗": int((~j.success_base & ~j.success_dec).sum())})
paired = pd.DataFrame(rows)
print("=== ペア比較（同じ初期状態） ===")
display(paired)
paired.to_csv(OUT_DIR / "paired_comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, title in [(axes[0], "success_rate", "Success rate"), (axes[1], "mean_progress", "Goal progress")]:
    by_task.pivot(index="task_id", columns="condition", values=col)[order].plot(kind="bar", ax=ax, ylim=(0, 1), rot=0)
    ax.set_title(title); ax.set_xlabel("LIBERO-10 task id"); ax.set_ylabel("")
plt.suptitle("SmolVLA on LIBERO-10: long instruction vs. LLM-decomposed subtasks")
plt.tight_layout(); plt.savefig(OUT_DIR / "comparison.png", dpi=150); plt.show()

# oracle_grounded と同じ3タスク（0, 1, 7）に絞った比較
sub = df[df["task_id"].isin(list(ORACLE))]
same3 = (sub.groupby("condition").agg(n=("success", "size"), success_rate=("success", "mean"),
                                      mean_progress=("progress", "mean"),
                                      first_goal_rate=("best_goals_satisfied", lambda s: (s >= 1).mean()))
           .reindex(order))
print("=== タスク0・1・7に絞った比較（first_goal_rate＝1つ目のゴールを達成した割合） ===")
display(same3.style.format({"success_rate": "{:.0%}", "mean_progress": "{:.0%}", "first_goal_rate": "{:.0%}"}))
same3.to_csv(OUT_DIR / "summary_tasks_0_1_7.csv")

## 12. 失敗パターンの確認（考察用）

In [ ]:
cols = ["task_id", "episode", "success", "best_goals_satisfied", "n_goals", "steps", "switches"]
display(df[df["condition"] == "oracle_grounded"][cols].sort_values(["task_id", "episode"]).reset_index(drop=True))
print("保存済み動画:", sorted(p.name for p in (OUT_DIR / "videos").glob("oracle_grounded_*.mp4")))